# 04 DPO：Verifier-guided Offline Preference Optimization

本节实现离线 DPO 主线：`SFT Policy → Rollout → Verifier → Chosen / Rejected → DPO`。偏好标签由确定性答案验证器产生，DPO Loss、Response Log Probability 和 Reference Policy 都在 Notebook 中显式实现。

当前 DGX Spark 阶段不加载模型、不启动 vLLM、不生成真实 Rollout。CPU Smoke Test 使用带有 `fixture=True` 标记的极小数值样本，只验证偏好构造和梯度方向，不把 Fixture 当作模型实验结果。

当前默认基座为 `Qwen/Qwen3-0.6B`，保留官方 Thinking 模板；0.6B GPU 基准与训练均待有卡后运行。配置与历史结果隔离方式见 [接入说明](docs/QWEN3_0_6B_SETUP.md)。

本次仅切换模型配置；本节仍使用旧版 Gravity / Equation Numeric / Bit 数据接口，尚未迁移到新版 Numeral 三任务。


## 学习目标

1. 记录多候选 Rollout 所需的 Token IDs、Response Mask、策略版本和截断状态；
2. 用独立 Verifier 选择最短正确 Chosen 与最短错误 Rejected；
3. 区分 DPO-A、Hard Negative 和 External Oracle 三类偏好构造；
4. 实现 Response Log Probability、Reference Log Probability 和标准 Sigmoid DPO Loss；
5. 验证 Reference Policy 不更新，且 DPO 的概率差异来自当前 Policy 更新。

## 1. 实验边界与控制变量

DPO 和后续 PPO/GRPO 都从同一个最佳 SFT Checkpoint 平行出发。Gravity 主要作为能力保持控制组；DPO 主线优先使用 Equation Numeric 和 Bit Manipulation。所有偏好对都必须记录 Verifier 版本、数据来源、策略版本和采样参数。

标准 DPO 目标：

\[
L_{DPO}=-\log\sigma\left(\beta\left[(\log\pi_\theta(y_w|x)-\log\pi_{ref}(y_w|x))- (\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_l|x))\right]\right)
\]

Reference Policy 固定为初始化的最佳 SFT 策略：`pi_ref = pi_SFT`。本 Notebook 不依赖 TRL 的 `DPOTrainer`。
输入沿用 00 保存的真实 Train，无需运行 01 合成实验。


In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "dpo"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "processed" / "checkpoints"
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
MODEL_ID = os.environ.get("QWEN3_MODEL_ID", "Qwen/Qwen3-0.6B")
MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1].lower().replace(".", "_")
DEFAULT_MODEL_PATH = Path.cwd() / "models" / MODEL_ID.rsplit("/", 1)[-1]
if not DEFAULT_MODEL_PATH.is_dir():
    DEFAULT_MODEL_PATH = Path.home() / "autodl-tmp/huggingface/models" / MODEL_ID.rsplit("/", 1)[-1]
MODEL_SOURCE = os.environ.get("QWEN3_MODEL_PATH", os.environ.get("LOCAL_MODEL_PATH",
    str(DEFAULT_MODEL_PATH) if DEFAULT_MODEL_PATH.is_dir() else MODEL_ID))
# Catch a stale 4B path before recording it under the 0.6B model ID.
if MODEL_ID == "Qwen/Qwen3-0.6B" and Path(MODEL_SOURCE).is_dir():
    _local_config = json.loads((Path(MODEL_SOURCE) / "config.json").read_text())
    if (_local_config.get("model_type"), _local_config.get("hidden_size"), _local_config.get("num_hidden_layers")) != ("qwen3", 1024, 28):
        raise ValueError("QWEN3_MODEL_PATH 与 Qwen3-0.6B 不匹配；请检查旧模型环境变量")
BASE_MODEL_PATH = MODEL_SOURCE
RESULT_DIR = RESULT_DIR / MODEL_TAG / os.environ.get("GPU_SMOKE_RUN_ID", "")
CHECKPOINT_ROOT = CHECKPOINT_ROOT / MODEL_TAG / os.environ.get("GPU_SMOKE_RUN_ID", "")
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
SEED = 42
RUN_ROLLOUT = os.environ.get("RUN_ROLLOUT", "0") == "1"  # backward-compatible umbrella flag
RUN_DPO = os.environ.get("RUN_DPO", "0") == "1"  # backward-compatible umbrella flag
RUN_DPO_ROLLOUT = os.environ.get("RUN_DPO_ROLLOUT", "1" if RUN_ROLLOUT else "0") == "1"
RUN_DPO_TRAINING = os.environ.get("RUN_DPO_TRAINING", "1" if RUN_DPO else "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")
DPO_ROLLOUT_PATH = RESULT_DIR / "dpo_smoke_rollouts.jsonl"
DPO_ROLLOUT_MANIFEST_PATH = RESULT_DIR / "dpo_rollout_manifest.json"
DPO_PREFERENCE_PATH = RESULT_DIR / "dpo_smoke_preferences.jsonl"
DPO_PREFERENCE_MANIFEST_PATH = RESULT_DIR / "dpo_preference_manifest.json"
SMOKE_PROMPTS_PER_TASK = 2
NUM_CANDIDATES_PER_PROMPT = 4
MAX_NEW_TOKENS = int(os.environ.get("RL_MAX_NEW_TOKENS", "2048"))
MAX_MODEL_LEN = int(os.environ.get("RL_MAX_MODEL_LEN", "8192"))
TEMPERATURE = 1.0
TOP_P = 1.0
DPO_BETA = 0.1
LEARNING_RATE = 1e-5
TRAINING_PROFILE = "official-pt-rl-cuda"

np.random.seed(SEED)
torch.manual_seed(SEED)

if not TRAIN_PATH.exists():
    raise FileNotFoundError("请先运行 00_Baseline_Diagnosis.ipynb，准备真实 Train 划分")
train_df = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
if train_df.empty:
    raise ValueError("真实 Train 划分为空")

def select_smoke_prompts(per_task: int = SMOKE_PROMPTS_PER_TASK) -> list[dict[str, object]]:
    selected = []
    for task in ("equation_numeric", "bit_manipulation"):
        for row in train_df[train_df["task_type"] == task].sample(n=min(per_task, int((train_df["task_type"] == task).sum())), random_state=SEED + (1 if task == "equation_numeric" else 2)).itertuples(index=False):
            selected.append({"id": str(row.id), "task_type": str(row.task_type), "prompt": str(row.prompt), "answer": str(row.answer), "source_split": "train"})
    return selected

print({"model_id": MODEL_ID, "real_train_rows": len(train_df), "run_rollout": RUN_ROLLOUT, "run_dpo": RUN_DPO})

## 2. 确定性 Verifier 与 Rollout Schema

答案正确不等于推理轨迹正确；本节先以最终答案作为 DPO-A 主线标签，并把 `trace_verified`、`source` 等字段保留下来，为 Process Preference 扩展留出接口。数值任务使用00_Baseline_Diagnosis.ipynb 的官方容差，Bit 使用 8-bit 精确匹配。

In [ ]:
OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v3"


def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    return __import__("re").sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=__import__("re").I).strip()


def extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    """Extract an explicitly marked final answer; never use an arbitrary CoT number."""
    import re
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?P<tag>final|answer)>\s*(?P<value>.*?)\s*</(?P=tag)>", text, flags=re.I | re.S)
    if tagged:
        return normalize_answer_text(tagged[-1][1]), "tagged"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        labeled = re.match(r"(?i)^(?:final answer|answer|最终答案|最终输出)\s*[:：]\s*(.+)$", line)
        if labeled:
            candidate = normalize_answer_text(labeled.group(1)).strip("`$")
            if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", candidate):
                return candidate.removeprefix("0b"), "final_label"
            if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", candidate):
                return candidate, "final_label"
        if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", line):
            return line.removeprefix("0b"), "standalone_final_line"
        if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", line):
            return line, "standalone_final_line"
    return None, "unmarked_or_empty"


def official_numeric_match(predicted: object, expected: object) -> bool:
    try:
        predicted_value, expected_value = float(normalize_answer_text(predicted)), float(normalize_answer_text(expected))
    except (TypeError, ValueError):
        return False
    return math.isfinite(predicted_value) and math.isfinite(expected_value) and math.isclose(predicted_value, expected_value, rel_tol=1e-2, abs_tol=1e-5)




#### 独立验证入口与诊断结果
答案提取要求明确格式；截断回答由完成状态检查单独拒绝，不从中间推导猜测最终答案。

In [ ]:
def verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text, expected_text = normalize_answer_text(predicted), normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p, e = predicted_text.removeprefix("0b"), expected_text.removeprefix("0b")
        if not __import__("re").fullmatch(r"[01]{8}", p) or not __import__("re").fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not __import__("re").fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        return official_numeric_match(predicted_text, expected_text), "official_math_isclose"
    return predicted_text == expected_text, "exact_text"


def extract_final_answer_for_rollout(text: str, task_type: str) -> str | None:
    return extract_final_answer(text, task_type)[0]


def verify_completion(task_type: str, completion: str, expected: str, truncated: bool = False) -> tuple[bool, str]:
    if truncated:
        return False, "truncated_output"
    predicted, _ = extract_final_answer(completion, task_type)
    return verify_answer(task_type, predicted, expected)


@dataclass
class RolloutRecord:
    sample_id: str
    task_type: str
    prompt: str
    completion: str
    expected_answer: str
    completion_token_ids: list[int]
    prompt_token_ids: list[int]
    response_mask: list[int]
    finish_reason: str
    truncated: bool
    policy_version: str
    verifier_version: str
    temperature: float
    top_p: float
    source: str
    fixture: bool = False
    trace_verified: bool = False
    correct: bool = False
    verification_method: str = ""


def record_to_dict(record: RolloutRecord) -> dict[str, object]:
    return {
        "sample_id": record.sample_id, "task_type": record.task_type, "prompt": record.prompt,
        "completion": record.completion, "expected_answer": record.expected_answer,
        "completion_token_ids": record.completion_token_ids, "prompt_token_ids": record.prompt_token_ids,
        "response_mask": record.response_mask, "finish_reason": record.finish_reason,
        "truncated": record.truncated, "policy_version": record.policy_version,
        "verifier_version": record.verifier_version, "temperature": record.temperature, "top_p": record.top_p,
        "source": record.source, "fixture": record.fixture, "trace_verified": record.trace_verified,
        "correct": record.correct, "verification_method": record.verification_method,
    }


## 3. CPU Fixture 和偏好对构造

CPU Fixture 为每个测试 Prompt 放入一条“正确格式答案”和一条“错误答案”，仅用于让选择逻辑可运行；它不代表模型采样，也不产生实验 Accuracy。真实运行时，`run_vllm_rollout` 必须替换 Fixture，并保存实际 Token IDs 和策略版本。

In [ ]:
def build_cpu_fixture_rollouts(samples: list[dict[str, object]], candidates: int = 4) -> list[dict[str, object]]:
    records = []
    for sample in samples[: SMOKE_PROMPTS_PER_TASK * 2]:
        task = str(sample["task_type"])
        answer = str(sample["answer"])
        good = f"Reasoning verified from all public examples. Final answer: \\boxed{{{answer}}}"
        wrong = "The examples were not fully checked. Final answer: \\boxed{0}"
        completions = [good, wrong] + [wrong + f" candidate {i}" for i in range(max(0, candidates - 2))]
        for index, completion in enumerate(completions):
            correct, method = verify_completion(task, completion, answer)
            records.append(record_to_dict(RolloutRecord(
                sample_id=str(sample.get("candidate_id", sample.get("id"))),
                task_type=task,
                prompt=str(sample["prompt"]),
                completion=completion,
                expected_answer=answer,
                completion_token_ids=list(range(1, len(completion.split()) + 1)),
                prompt_token_ids=list(range(100, 100 + len(str(sample["prompt"]).split()))),
                response_mask=[1] * len(completion.split()),
                finish_reason="stop",
                truncated=False,
                policy_version="cpu_fixture",
                verifier_version=OFFICIAL_VERIFIER_VERSION,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                source="cpu_fixture_only",
                fixture=True,
                trace_verified=True,
                correct=correct,
                verification_method=method,
            )))
    return records


def _completion_length(record: dict[str, object]) -> int:
    return len(record.get("completion_token_ids", []))


def construct_preference_pairs(rollouts: list[dict[str, object]], mode: str = "dpo_a") -> list[dict[str, object]]:
    """Construct Chosen/Rejected pairs after independent verification."""
    if mode not in {"dpo_a", "hard_negative", "external_oracle"}:
        raise ValueError(mode)
    grouped = defaultdict(list)
    for record in rollouts:
        grouped[(record["sample_id"], record["task_type"])].append(record)
    pairs = []
    for (sample_id, task_type), group in grouped.items():
        correct = [row for row in group if row.get("correct") and not row.get("truncated")]
        incorrect = [row for row in group if not row.get("correct") and not row.get("truncated")]
        if mode == "external_oracle":
            oracle = next((row for row in group if row.get("source") == "deterministic_solver_oracle" and row.get("correct") and not row.get("truncated")), None)
            if oracle is not None:
                correct = [oracle]
        if not correct or not incorrect:
            continue
        chosen = min(correct, key=_completion_length)
        rejected = min(incorrect, key=_completion_length)
        pairs.append({
            "sample_id": sample_id,
            "task_type": task_type,
            "prompt": chosen["prompt"],
            "chosen": chosen["completion"],
            "rejected": rejected["completion"],
            "prompt_token_ids": chosen["prompt_token_ids"],
            "chosen_token_ids": chosen["completion_token_ids"],
            "rejected_token_ids": rejected["completion_token_ids"],
            "chosen_correct": True,
            "rejected_correct": False,
            "chosen_trace_verified": bool(chosen.get("trace_verified")),
            "chosen_truncated": bool(chosen.get("truncated")),
            "rejected_truncated": bool(rejected.get("truncated")),
            "preference_mode": mode,
            "verifier_version": chosen["verifier_version"],
            "source": "rollout_verifier",
            "fixture": bool(chosen.get("fixture", False)),
        })
    return pairs


def audit_preference_pairs(pairs: list[dict[str, object]]) -> dict[str, object]:
    reasons = Counter()
    valid = []
    for pair in pairs:
        checks = {
            "chosen_correct": pair.get("chosen_correct") is True,
            "rejected_incorrect": pair.get("rejected_correct") is False,
            "different_text": pair.get("chosen") != pair.get("rejected"),
            "same_prompt": bool(pair.get("prompt")),
            "not_truncated_chosen": not pair.get("chosen_truncated", False) and not pair.get("rejected_truncated", False),
        }
        if all(checks.values()):
            valid.append(pair)
        else:
            reasons.update(name for name, ok in checks.items() if not ok)
    return {"input_pairs": len(pairs), "valid_pairs": len(valid), "rejection_reasons": dict(reasons), "fixture_pairs": int(sum(pair.get("fixture", False) for pair in valid))}


fixture_rollouts = build_cpu_fixture_rollouts(select_smoke_prompts(), NUM_CANDIDATES_PER_PROMPT)
preference_pairs = construct_preference_pairs(fixture_rollouts, "dpo_a")
print({"fixture_rollouts": len(fixture_rollouts), "preference_pairs": len(preference_pairs), "audit": audit_preference_pairs(preference_pairs)})

## 4. RL Readiness Audit（DPO 前置诊断）

DPO 不要求 GRPO 式组内方差，但同一批 Rollout 仍应报告 All-correct / Mixed / All-wrong 覆盖，避免把极少数偏好对误认为充分训练信号。

In [ ]:
def rl_readiness_audit(rollouts: list[dict[str, object]], group_size: int = NUM_CANDIDATES_PER_PROMPT) -> dict[str, object]:
    grouped = defaultdict(list)
    for row in rollouts:
        grouped[(row["sample_id"], row["task_type"])].append(int(bool(row.get("correct"))))
    categories = Counter()
    variances = []
    for rewards in grouped.values():
        rewards = rewards[:group_size]
        correct_count = sum(rewards)
        if correct_count == len(rewards): categories["all_correct"] += 1
        elif correct_count == 0: categories["all_wrong"] += 1
        else: categories["mixed"] += 1
        variances.append(float(np.var(rewards)))
    total = sum(categories.values())
    return {
        "groups": total,
        "categories": dict(categories),
        "all_correct_ratio": categories["all_correct"] / total if total else 0.0,
        "mixed_ratio": categories["mixed"] / total if total else 0.0,
        "all_wrong_ratio": categories["all_wrong"] / total if total else 0.0,
        "mean_reward_variance": float(np.mean(variances)) if variances else 0.0,
        "fixture_only": bool(rollouts) and all(row.get("fixture", False) for row in rollouts),
    }


print(json.dumps({"rl_readiness": rl_readiness_audit(fixture_rollouts)}, ensure_ascii=False, indent=2))

## 5. Response Log Probability 与 DPO Loss

只对 Completion Token 计算序列 Log Probability；Prompt Token、Padding 和截断后的无效位置都不参与。DPO 的 Chosen / Rejected 使用同一 Prompt，Reference Log Probability 在训练期间 `no_grad` 且参数冻结。

In [ ]:
def token_logprobs_from_logits(logits: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
    """Return log p(token_t | token_<t) with shape [batch, time-1]."""
    if logits.ndim != 3 or input_ids.ndim != 2 or logits.shape[:2] != input_ids.shape:
        raise ValueError("logits [B,T,V] and input_ids [B,T] must align")
    log_probs = F.log_softmax(logits[:, :-1, :].float(), dim=-1)
    return log_probs.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)


def sequence_logprob(logits: torch.Tensor, input_ids: torch.Tensor, response_mask: torch.Tensor, length_normalize: bool = False) -> torch.Tensor:
    token_logprobs = token_logprobs_from_logits(logits, input_ids)
    mask = response_mask[:, 1:].to(token_logprobs.dtype)
    values = (token_logprobs * mask).sum(dim=-1)
    if length_normalize:
        values = values / mask.sum(dim=-1).clamp_min(1.0)
    return values


def dpo_loss(
    policy_chosen_logp: torch.Tensor,
    policy_rejected_logp: torch.Tensor,
    reference_chosen_logp: torch.Tensor,
    reference_rejected_logp: torch.Tensor,
    beta: float = DPO_BETA,
) -> dict[str, torch.Tensor]:
    chosen_advantage = policy_chosen_logp - reference_chosen_logp
    rejected_advantage = policy_rejected_logp - reference_rejected_logp
    logits = beta * (chosen_advantage - rejected_advantage)
    loss = -F.logsigmoid(logits).mean()
    return {
        "loss": loss,
        "preference_logits": logits.detach(),
        "chosen_reward_margin": (beta * chosen_advantage).detach(),
        "rejected_reward_margin": (beta * rejected_advantage).detach(),
    }


def dpo_step(
    policy: nn.Module,
    reference: nn.Module,
    batch: dict[str, torch.Tensor],
    optimizer: torch.optim.Optimizer,
    beta: float = DPO_BETA,
) -> float:
    policy.train()
    reference.eval()
    policy_chosen = policy(batch["chosen_input_ids"])["logits"]
    policy_rejected = policy(batch["rejected_input_ids"])["logits"]
    with torch.no_grad():
        reference_chosen = reference(batch["chosen_input_ids"])["logits"]
        reference_rejected = reference(batch["rejected_input_ids"])["logits"]
    values = dpo_loss(
        sequence_logprob(policy_chosen, batch["chosen_input_ids"], batch["chosen_response_mask"]),
        sequence_logprob(policy_rejected, batch["rejected_input_ids"], batch["rejected_response_mask"]),
        sequence_logprob(reference_chosen, batch["chosen_input_ids"], batch["chosen_response_mask"]),
        sequence_logprob(reference_rejected, batch["rejected_input_ids"], batch["rejected_response_mask"]),
        beta,
    )
    optimizer.zero_grad(set_to_none=True)
    values["loss"].backward()
    optimizer.step()
    return float(values["loss"].detach())

## 6. GPU Rollout 和 DPO 训练入口（当前关闭）

真实运行时先激活独立 vLLM Profile 生成每题 4 条候选，保存实际 Completion Token IDs / Logprobs / 截断状态，再关闭 vLLM 切换训练 Profile。下面的入口明确区分 Rollout 和 DPO 更新；CPU 阶段不会触发任何导入或模型分配。

### 6.1 读取并校验 SFT Manifest

In [ ]:
def read_sft_manifest(path: Path = SFT_MANIFEST_PATH) -> dict[str, object]:
    if not path.exists():
        raise FileNotFoundError(f"SFT Manifest 不存在：{path}；请先运行 Notebook 03")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != "nemotron-sft-checkpoint/v1":
        raise ValueError("Unsupported SFT manifest schema")
    if manifest.get("base_model", {}).get("id") != MODEL_ID:
        raise ValueError("SFT Manifest 的 Base Model 与课程固定模型不一致")
    if manifest.get("status") != "completed":
        raise ValueError("SFT Manifest 尚未标记为 completed，不能初始化 DPO")
    adapter_path = Path(resolve_manifest_path(str(manifest.get("adapter_path", ""))))
    if not adapter_path.is_dir() or not (adapter_path / "adapter_config.json").exists():
        raise FileNotFoundError(f"SFT Adapter 不存在或缺少 adapter_config.json：{adapter_path}")
    return manifest


def resolve_manifest_path(value: str) -> str:
    path = Path(value)
    return str(path if path.is_absolute() else PROJECT_ROOT / path)


### 6.2 解析 vLLM 行为输出的 Token Logprob

### 6.3 vLLM Rollout：一次生成多条候选

In [ ]:
def run_vllm_rollout(prompt_rows, model_source, policy_version, adapter_path=None):
    records = _generate_rollout_records(prompt_rows, model_source or BASE_MODEL_PATH, policy_version, adapter_path, NUM_CANDIDATES_PER_PROMPT)
    for row in records:
        row["correct"], row["verification_method"] = verify_completion(row["task_type"], row["completion"], row["expected_answer"], row["truncated"])
        row["verifier_version"] = OFFICIAL_VERIFIER_VERSION
    return records


### 6.4 Chosen/Rejected 的 Token 化与 Padding

In [ ]:
def _encode_preference_pair(tokenizer, prompt: str, completion: str, max_length: int = 2048) -> dict[str, list[int]]:
    messages, assistant = [{"role": "user", "content": prompt}], {"role": "assistant", "content": completion}
    try:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False, enable_thinking=True)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(messages + [assistant], tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False).input_ids
    if len(full_ids) < len(prompt_ids) or full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("preference Chat Template 的 Prompt 前缀不一致")
    full_ids = full_ids[:max_length]
    if len(full_ids) <= len(prompt_ids):
        raise ValueError("preference completion 被截断为空")
    return {"input_ids": full_ids, "response_mask": [0] * len(prompt_ids) + [1] * (len(full_ids) - len(prompt_ids))}


def _collate_preference_examples(examples: list[dict[str, list[int]]], pad_token_id: int) -> dict[str, torch.Tensor]:
    max_len = max(max(len(example["chosen_input_ids"]), len(example["rejected_input_ids"])) for example in examples)
    values = {"chosen_input_ids": [], "rejected_input_ids": [], "chosen_response_mask": [], "rejected_response_mask": []}
    for example in examples:
        for prefix in ("chosen", "rejected"):
            ids, mask = example[f"{prefix}_input_ids"], example[f"{prefix}_response_mask"]
            pad = max_len - len(ids)
            values[f"{prefix}_input_ids"].append(ids + [pad_token_id] * pad)
            values[f"{prefix}_response_mask"].append(mask + [0] * pad)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in values.items()}


### 6.5 从同一 SFT Adapter 初始化 Policy 与冻结 Reference

In [ ]:
def _load_sft_policy_and_reference(manifest: dict, mode: str, load_reference: bool = True):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training
    kwargs = {"device_map": {"": 0}, "attn_implementation": "sdpa", "torch_dtype": torch.bfloat16}
    if mode == "nf4_qlora":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    elif mode != "bf16_lora": raise ValueError(mode)
    def load(trainable):
        base = AutoModelForCausalLM.from_pretrained(manifest["base_model"]["path"], **kwargs)
        base.config.use_cache = False
        if mode == "nf4_qlora" and trainable:
            base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
        policy = PeftModel.from_pretrained(base, resolve_manifest_path(manifest["adapter_path"]), is_trainable=trainable)
        policy.eval()  # Turn off LoRA dropout; eval still supports gradient propagation.
        if trainable:
            policy.gradient_checkpointing_enable()
            policy.enable_input_require_grads()
        return policy
    tokenizer = AutoTokenizer.from_pretrained(manifest.get("tokenizer_path", manifest["base_model"]["path"]))
    return load(True), load(False) if load_reference else None, tokenizer, resolve_manifest_path(manifest["adapter_path"])


### 6.6 Adapter 产物可重载性检查

In [ ]:
def validate_saved_adapter_artifact(adapter_path: Path) -> dict[str, object]:
    config = adapter_path / "adapter_config.json"
    weights = list(adapter_path.glob("adapter_model.*"))
    return {"path": str(adapter_path), "config_exists": config.exists(), "weights_exists": bool(weights), "reloadable_artifact": config.exists() and bool(weights)}


### 6.7 DPO Update 与 Adapter 保存

In [ ]:
def _reload_dpo_adapter(base_path: str, adapter_path: Path, mode: str) -> dict[str, object]:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    if not validate_saved_adapter_artifact(adapter_path)["reloadable_artifact"]:
        return {"status": "failed_artifact_missing"}
    base = None
    try:
        kwargs = {"trust_remote_code": True, "device_map": "auto"}
        if mode == "nf4_qlora":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
        elif mode == "bf16_lora":
            kwargs["torch_dtype"] = torch.bfloat16
        else:
            raise ValueError(mode)
        base = AutoModelForCausalLM.from_pretrained(base_path, **kwargs)
        reloaded = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)
        return {"status": "passed", "trainable_parameter_tensors": int(sum(p.requires_grad for p in reloaded.parameters()))}
    except torch.cuda.OutOfMemoryError as exc:
        return {"status": "oom", "error": repr(exc)}
    except Exception as exc:
        return {"status": "failed", "error": repr(exc)}
    finally:
        if base is not None:
            del base
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


def run_dpo_training(preference_pairs, sft_manifest_path=SFT_MANIFEST_PATH, mode="nf4_qlora"):
    if not RUN_DPO_TRAINING: return {"status":"skipped_by_default"}
    if not preference_pairs: return {"status":"blocked_no_preference_pairs", "optimizer_steps":0}
    manifest = read_sft_manifest(sft_manifest_path)
    policy, _, tokenizer, parent_adapter = _load_sft_policy_and_reference(manifest, mode, load_reference=False)
    def score(ids, mask):
        output = policy(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)
        return sequence_logprob(output.logits.float(), ids, mask)
    device = policy.get_input_embeddings().weight.device
    examples, cached = [], []
    for pair in preference_pairs:
        prompt = list(pair["prompt_token_ids"])
        sides = []
        for name in ("chosen", "rejected"):
            response = list(pair[name + "_token_ids"])
            if not response or len(prompt) + len(response) > MAX_MODEL_LEN:
                raise ValueError("DPO pair exceeds context; no silent preference truncation")
            sides.append((torch.tensor([prompt + response], device=device), torch.tensor([[0]*len(prompt)+[1]*len(response)], device=device)))
        with torch.no_grad():
            reference_chosen, reference_rejected = (score(ids, mask).detach() for ids, mask in sides)
        examples.append(sides)
        cached.append({"sample_id":pair["sample_id"], "chosen":float(reference_chosen), "rejected":float(reference_rejected), "reference":"frozen_initial_sft", "mode":mode})
    _write_stage_json(RESULT_DIR / "dpo_reference_logprobs.json", {"status":"completed", "sft_fingerprint":_sft_fingerprint(manifest), "rows":cached})
    # Check the unchanged initial SFT against the immutable no_grad cache.
    with torch.no_grad():
        initial = score(*examples[0][0])
    initialization_diff = abs(float(initial) - cached[0]["chosen"])
    if initialization_diff > 1e-3: raise ValueError("DPO initial reference cache mismatch")
    policy.train()
    for module in policy.modules():
        if isinstance(module, nn.Dropout): module.eval()
    optimizer = torch.optim.AdamW((p for p in policy.parameters() if p.requires_grad), lr=LEARNING_RATE)
    losses = []
    for sides, frozen in zip(examples, cached):
        chosen, rejected = (score(ids, mask) for ids, mask in sides)
        loss = dpo_loss(chosen, rejected, chosen.new_tensor([frozen["chosen"]]), rejected.new_tensor([frozen["rejected"]]), DPO_BETA)["loss"]
        if not torch.isfinite(loss): raise ValueError("Non-finite DPO loss")
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step(); losses.append(float(loss.detach()))
        if RUN_MODE == "smoke": break  # One actual update; do not silently train all preference pairs.
    output_dir = CHECKPOINT_ROOT / f"dpo_{mode}"
    policy.save_pretrained(output_dir); tokenizer.save_pretrained(output_dir)
    reload_check = actual_adapter_reload(policy, output_dir, preference_pairs[0]["prompt_token_ids"])
    result = {"status":"completed", "optimizer_steps":len(losses), "losses":losses, "pairs":len(preference_pairs),
        "reference_policy":"immutable logprob cache of identical initial SFT Adapter", "reference_initialization_max_abs_diff":initialization_diff,
        "parent_sft_fingerprint":_sft_fingerprint(manifest), "parent_sft_adapter":manifest["adapter_path"], "base_model":manifest["base_model"], "tokenizer_path":manifest.get("tokenizer_path"),
        "adapter_path":str(output_dir.relative_to(PROJECT_ROOT)), "adapter_reload":reload_check, "mode":mode}
    _write_stage_json(output_dir / "dpo_manifest.json", result)
    return result


### 6.8 DPO Smoke Pipeline：Rollout → Verifier → Preference → Update

In [ ]:
def _write_dpo_jsonl(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in rows) + ("\n" if rows else ""), encoding="utf-8")


def _read_dpo_jsonl(path: Path) -> list[dict[str, object]]:
    if not path.exists():
        raise FileNotFoundError(f"阶段产物不存在：{path}")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _dpo_gpu_preflight(stage: str) -> dict[str, object]:
    if not torch.cuda.is_available():
        return {"status": "blocked_no_cuda", "stage": stage}
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    result = {"status": "ready", "stage": stage, "device": torch.cuda.get_device_name(0), "free_memory_mib": free_bytes / 2**20, "total_memory_mib": total_bytes / 2**20}
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    (RESULT_DIR / "gpu_preflight.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def run_dpo_rollout_phase():
    if not RUN_DPO_ROLLOUT: return {"status":"skipped_by_default"}
    def operation():
        manifest = read_sft_manifest(); prompts = select_smoke_prompts()
        rows = run_vllm_rollout(prompts, manifest["base_model"]["path"], manifest["policy_version"], resolve_manifest_path(manifest["adapter_path"]))
        return _persist_rollout(rows, DPO_ROLLOUT_PATH, DPO_ROLLOUT_MANIFEST_PATH, manifest, len(prompts)*NUM_CANDIDATES_PER_PROMPT)
    return _gpu_stage("dpo_rollout", operation)


def run_dpo_training_phase(mode="nf4_qlora"):
    if not RUN_DPO_TRAINING: return {"status":"skipped_by_default"}
    def operation():
        rows = _read_rollout(DPO_ROLLOUT_PATH, DPO_ROLLOUT_MANIFEST_PATH)
        for row in rows:
            row["correct"], row["verification_method"] = verify_completion(row["task_type"], row["completion"], row["expected_answer"], row["truncated"])
        for row in rows: row["verifier_version"] = OFFICIAL_VERIFIER_VERSION
        _write_dpo_jsonl(RESULT_DIR / "dpo_verified_rollouts.jsonl", rows)
        pairs = construct_preference_pairs(rows, "dpo_a")
        _write_dpo_jsonl(DPO_PREFERENCE_PATH, pairs)
        audit = audit_preference_pairs(pairs)
        _write_stage_json(DPO_PREFERENCE_MANIFEST_PATH, {"status":"completed", "preference_audit":audit, "rollouts":len(rows), "truncated":sum(r["truncated"] for r in rows)})
        result = run_dpo_training(pairs, SFT_MANIFEST_PATH, mode)
        return {**result, "preference_audit":audit}
    return _gpu_stage("dpo_training", operation)


def run_dpo_smoke_pipeline(mode="nf4_qlora"):
    if RUN_DPO_ROLLOUT and RUN_DPO_TRAINING:
        return {"status":"blocked_kernel_switch_required", "reason":"Run Rollout and Training in separate profiles/processes"}
    if RUN_DPO_ROLLOUT: return run_dpo_rollout_phase()
    if RUN_DPO_TRAINING: return run_dpo_training_phase(mode)
    return {"status":"skipped_by_default"}


### 6.9 GPU Rollout / DPO 开关

In [ ]:
if RUN_DPO_ROLLOUT or RUN_DPO_TRAINING:
    print({"gpu_entry": "enabled_when_explicitly_requested", "rollout": RUN_DPO_ROLLOUT, "training": RUN_DPO_TRAINING, "training_profile": TRAINING_PROFILE})
else:
    print({"rollout": "disabled_by_default", "dpo_training": "disabled_by_default", "reference_policy": "same_sft_adapter"})


## 7. CPU Smoke Test：偏好质量、Log Probability 和梯度

用两个小型语言模型和手工 Token 序列验证：Chosen/Rejected 标签正确、Reference 不产生梯度、DPO Loss 有限且当前 Policy 获得梯度。测试通过不代表真实 DPO 已运行。

In [ ]:
class TinyLM(nn.Module):
    def __init__(self, vocab_size: int = 24, hidden_size: int = 12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)
    def forward(self, input_ids):
        return {"logits": self.head(self.embedding(input_ids))}


def make_toy_preference_batch(batch_size: int = 2, time_steps: int = 6, vocab_size: int = 24) -> dict[str, torch.Tensor]:
    torch.manual_seed(SEED)
    chosen = torch.randint(0, vocab_size, (batch_size, time_steps))
    rejected = torch.randint(0, vocab_size, (batch_size, time_steps))
    mask = torch.zeros(batch_size, time_steps, dtype=torch.long)
    mask[:, 2:] = 1
    return {
        "chosen_input_ids": chosen,
        "rejected_input_ids": rejected,
        "chosen_response_mask": mask,
        "rejected_response_mask": mask.clone(),
    }


policy = TinyLM()
reference = TinyLM()
reference.load_state_dict(policy.state_dict())
for parameter in reference.parameters():
    parameter.requires_grad_(False)
optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-2)
batch = make_toy_preference_batch()
old_reference = [parameter.detach().clone() for parameter in reference.parameters()]
loss_value = dpo_step(policy, reference, batch, optimizer)
assert math.isfinite(loss_value)
assert all(parameter.grad is None for parameter in reference.parameters())
assert any(parameter.grad is not None for parameter in policy.parameters())
assert all(torch.equal(before, after) for before, after in zip(old_reference, reference.parameters()))
assert audit_preference_pairs(preference_pairs)["valid_pairs"] == len(preference_pairs)
# Explicit final markers win over unrelated numbers in the Thinking trace.
assert verify_completion("gravity", "Step 1 uses t=2.0; Step 2 uses 7.0.\n\\boxed{42.0}", "42.0")[0]
assert verify_completion("gravity", "Step 1 uses 2.0; final token budget is 99", "42.0")[0] is False
assert verify_completion("gravity", "\\boxed{42.0}", "42.0", truncated=True)[1] == "truncated_output"
# Standard DPO keeps sequence log-probability as a sum, so a longer response
# contributes more token log-probability without length normalization.
zero_short = torch.zeros(1, 3, 24)
zero_long = torch.zeros(1, 4, 24)
short_ids = torch.tensor([[1, 2, 3]])
long_ids = torch.tensor([[1, 2, 3, 4]])
short_mask = torch.tensor([[0, 1, 1]])
long_mask = torch.tensor([[0, 1, 1, 1]])
short_logp = sequence_logprob(zero_short, short_ids, short_mask)
long_logp = sequence_logprob(zero_long, long_ids, long_mask)
assert long_logp.item() < short_logp.item()

dpo_cpu_report = {
    "status": "passed",
    "fixture_rollouts": len(fixture_rollouts),
    "fixture_preference_pairs": len(preference_pairs),
    "preference_audit": audit_preference_pairs(preference_pairs),
    "readiness_audit": rl_readiness_audit(fixture_rollouts),
    "dpo_loss_finite": True,
    "reference_frozen": True,
    "reference_initialization_match": True,
    "verifier_version": OFFICIAL_VERIFIER_VERSION,
    "policy_gradient": True,
    "gpu_rollout": "not_run",
    "gpu_dpo": "not_run",
}
print(json.dumps(dpo_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与限制

CPU 阶段完成了偏好数据结构、Verifier 标签、RL Readiness Audit、Response Log Probability、Reference Freeze 和 DPO 梯度 Smoke Test。真实 Rollout、Adapter 训练、保存/重载和 SFT→DPO 指标仍是 `Not Run`，不能把 Fixture 对作为实验结果。

明天必须先选定并固定最佳 SFT Checkpoint，再在 RTX 5090 上对 Equation Numeric / Bit Manipulation 生成少量真实候选，统计偏好对产生率和标签质量，最后运行小步 DPO。若偏好对过少，应记录为数据覆盖问题，而不是强行训练。

## 8. SFT Manifest 交接检查

真实 DPO 更新只在 `RUN_DPO=True` 时执行；CPU 阶段检查入口会读取同一 SFT Manifest，Adapter 目录不再直接传给 `AutoModelForCausalLM`。


In [ ]:
if SFT_MANIFEST_PATH.exists():
    dpo_manifest_probe = read_sft_manifest(SFT_MANIFEST_PATH)
    assert dpo_manifest_probe["base_model"]["id"] == MODEL_ID
    dpo_manifest_status = "available"
else:
    dpo_manifest_status = "not_run_sft_manifest_missing"
if not (RUN_DPO_ROLLOUT or RUN_DPO_TRAINING):
    assert RUN_DPO_ROLLOUT is False and RUN_DPO_TRAINING is False
dpo_checkpoint_report = {"status":"passed", "smoke_prompts": len(select_smoke_prompts()), "candidates_per_prompt": NUM_CANDIDATES_PER_PROMPT, "preference_pairs": len(preference_pairs), "reference_frozen": True, "reference_initialization_match": True, "sequence_logprob_is_sum": True, "verifier_version": OFFICIAL_VERIFIER_VERSION, "sft_manifest": dpo_manifest_status, "gpu": "not_run"}
(RESULT_DIR / "cpu_dpo_smoke_report.json").parent.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "cpu_dpo_smoke_report.json").write_text(json.dumps(dpo_checkpoint_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(dpo_checkpoint_report, ensure_ascii=False, indent=2))


### GPU 阶段：资源、失败与状态记录
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def _write_stage_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def _gpu_stage(stage: str, operation) -> dict:
    """Preflight + device memory sampler + truthful failure/status report."""
    import gc, threading, subprocess, traceback, importlib.metadata
    started = time.perf_counter()
    stop, samples = threading.Event(), []
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    report_path = RESULT_DIR / f"{stage}_status.json"
    if not torch.cuda.is_available():
        result = {"status": "blocked_no_cuda", "stage": stage}
        _write_stage_json(report_path, result)
        return result
    free, total = torch.cuda.mem_get_info()
    preflight = {"device": torch.cuda.get_device_name(0), "total_mib": total / 2**20,
                 "free_mib": free / 2**20, "torch_cuda": torch.version.cuda,
                 "bf16_supported": torch.cuda.is_bf16_supported()}
    preflight["versions"] = {}
    for package in ("torch", "transformers", "peft", "bitsandbytes", "vllm"):
        try: preflight["versions"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: pass
    _write_stage_json(RESULT_DIR / f"{stage}_preflight.json", preflight)
    def monitor():
        while not stop.is_set():
            try:
                output = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True, timeout=3)
                samples.append(float(output.splitlines()[0]))
            except (OSError, ValueError, subprocess.SubprocessError): pass
            stop.wait(0.5)
    thread = threading.Thread(target=monitor, daemon=True)
    torch.cuda.reset_peak_memory_stats()
    thread.start()
    try:
        if "training" in stage and "vllm" in __import__("sys").modules:
            result = {"status": "blocked_kernel_switch_required", "reason": "先结束 vLLM 进程并切换训练 Kernel"}
        elif free / total < 0.80:
            result = {"status": "blocked_gpu_occupied", "reason": "GPU 可用显存不足 80%；不终止其他任务"}
        else:
            result = operation()
            if not isinstance(result, dict) or "status" not in result:
                raise ValueError("阶段必须显式返回 status")
    except Exception as exc:
        message = traceback.format_exc()
        result = {"status": "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower() else "failed",
                  "error": repr(exc), "traceback": message}
        (RESULT_DIR / f"{stage}_error.log").write_text(message, encoding="utf-8")
    finally:
        stop.set(); thread.join(timeout=4)
        gc.collect(); torch.cuda.empty_cache()
    result.update({"stage": stage, "wall_time_seconds": time.perf_counter() - started,
                   "preflight": preflight, "peak_memory_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,
                   "peak_device_memory_mib": max(samples) if samples else None})
    _write_stage_json(report_path, result)
    return result


### 阶段交接：真实 Token、原始 Logprob 与 Adapter 指纹
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def _file_sha(path: Path) -> str:
    import hashlib
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _sft_fingerprint(manifest: dict) -> dict:
    adapter = Path(resolve_manifest_path(manifest["adapter_path"]))
    return {"base_model_id": manifest["base_model"]["id"], "policy_version": manifest["policy_version"],
            "manifest_sha256": _file_sha(SFT_MANIFEST_PATH),
            "adapter_files": {p.name: _file_sha(p) for p in sorted(adapter.glob("adapter_*")) if p.is_file()}}


def _chosen_logprob_values(generated) -> list[float]:
    """vLLM 0.11 chosen-token dictionary lookup; never substitute another token."""
    raw, ids = generated.logprobs, list(generated.token_ids or [])
    if raw is None or not ids or len(raw) != len(ids):
        raise ValueError("Missing or misaligned vLLM token logprobs")
    result = []
    for token, candidates in zip(ids, raw):
        if not isinstance(candidates, dict) or token not in candidates:
            raise ValueError(f"Chosen token {token} missing from vLLM Logprob dictionary")
        value = float(candidates[token].logprob)
        if not math.isfinite(value): raise ValueError("non-finite token logprob")
        result.append(value)
    return result


def _persist_rollout(records: list[dict], path: Path, manifest_path: Path, sft: dict, expected_count: int) -> dict:
    if len(records) != expected_count: raise ValueError("Rollout count differs from Smoke budget")
    fingerprint = _sft_fingerprint(sft)
    _check_train_origin(records)
    for row in records:
        if row.get("fixture"): raise ValueError("CPU fixture cannot enter GPU pipeline")
        ids, completion = row["prompt_token_ids"], row["completion_token_ids"]
        if not ids or not completion or len(completion) != len(row["behavior_logprobs"]):
            raise ValueError("Invalid token alignment")
        if row["response_mask"] != [0] * len(ids) + [1] * len(completion):
            raise ValueError("Response Mask differs from stored Token IDs")
        row.update({"source_split": "train", "rollout_quantization": "bf16", "logprob_semantics": "vllm_v1_raw_logprobs",
                    "sampling_distribution": "raw_softmax_T1_topP1_topK_disabled", "sft_fingerprint": fingerprint})
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records), encoding="utf-8")
    payload = {"schema_version": "nemotron-rollout/v2", "status": "completed", "records": len(records),
               "rollout_path": str(path.relative_to(PROJECT_ROOT)), "rollout_sha256": _file_sha(path),
               "sft_fingerprint": fingerprint, "source_split": "train", "temperature": 1.0, "top_p": 1.0,
               "top_k": -1, "logprobs_mode": "raw_logprobs", "quantization": "bf16",
               "max_new_tokens": MAX_NEW_TOKENS, "max_model_len": MAX_MODEL_LEN}
    _write_stage_json(manifest_path, payload)
    return payload


def _read_rollout(path: Path, manifest_path: Path) -> list[dict]:
    saved = json.loads(manifest_path.read_text())
    if saved.get("status") != "completed" or saved.get("rollout_sha256") != _file_sha(path):
        raise ValueError("Incomplete or modified Rollout artifact")
    if saved.get("sft_fingerprint") != _sft_fingerprint(read_sft_manifest()):
        raise ValueError("SFT Manifest/Adapter changed since Rollout; refuse policy-version mismatch")
    records = [json.loads(s) for s in path.read_text().splitlines() if s.strip()]
    if len(records) != saved["records"]: raise ValueError("Rollout record count mismatch")
    _check_train_origin(records)
    expected_fingerprint = saved["sft_fingerprint"]
    seen = set()
    for row in records:
        key = (row["sample_id"], row["group_index"])
        if key in seen: raise ValueError("Duplicate candidate index in Rollout group")
        seen.add(key)
        if row.get("sft_fingerprint") != expected_fingerprint or row.get("policy_version") != expected_fingerprint["policy_version"]:
            raise ValueError("Per-record SFT policy version mismatch")
        ids, completion = row["prompt_token_ids"], row["completion_token_ids"]
        if not ids or not completion or len(completion) != len(row["behavior_logprobs"]):
            raise ValueError("Invalid stored token/logprob alignment")
        if row["response_mask"] != [0]*len(ids)+[1]*len(completion):
            raise ValueError("Stored Response Mask mismatch")
        if not all(math.isfinite(float(x)) and float(x)<=1e-6 for x in row["behavior_logprobs"]):
            raise ValueError("Invalid sampled-token log probability")
        if row.get("source_split") != "train" or row.get("fixture"):
            raise ValueError("Only real Train Rollout may train")
        if row.get("temperature") != 1.0 or row.get("top_p") != 1.0:
            raise ValueError("This Smoke requires T=1, top_p=1 for raw/behavior logprob equivalence")
    return records


### 策略对齐：BF16 Rollout 与训练侧量化差异
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def audit_policy_logprobs(model, tokenizer, records: list[dict], mode: str) -> dict:
    """Compare actual generated Token IDs without re-tokenizing the completion.

    One real sequence per task is teacher-forced. NF4 is explicitly a different
    target policy from the BF16 rollout; old denominator logprobs stay unchanged.
    """
    chosen, seen = [], set()
    for row in records:
        if row["task_type"] not in seen:
            chosen.append(row); seen.add(row["task_type"])
    deltas, checks = [], []
    model.eval()
    device = model.get_input_embeddings().weight.device
    for row in chosen:
        prompt_ids, response_ids = row["prompt_token_ids"], row["completion_token_ids"]
        rendered_ids = tokenizer.apply_chat_template([{"role":"user", "content":row["prompt"]}], tokenize=True, add_generation_prompt=True, enable_thinking=True)
        # Transformers 4 returns list[int]; Transformers 5 defaults to BatchEncoding.
        if hasattr(rendered_ids, "keys"):
            rendered_ids = rendered_ids["input_ids"]
        if list(rendered_ids) != prompt_ids: raise ValueError("Stored vLLM Prompt Token IDs differ from official Chat Template")
        if row["response_mask"] != [0] * len(prompt_ids) + [1] * len(response_ids):
            raise ValueError("Prompt/Response mask mismatch")
        ids = torch.tensor([prompt_ids + response_ids], device=device)
        with torch.no_grad():
            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)
            logits = output.logits[0]
            start = len(prompt_ids) - 1
            values = []
            for offset in range(0, len(response_ids), 64):
                count = min(64, len(response_ids) - offset)
                block = logits[start + offset:start + offset + count].float()
                labels = ids[0, start + offset + 1:start + offset + count + 1]
                selected = block.gather(-1, labels[:, None]).squeeze(-1) - torch.logsumexp(block, dim=-1)
                values.extend(selected.cpu().tolist())
        delta = torch.tensor(values) - torch.tensor(row["behavior_logprobs"], dtype=torch.float32)
        deltas.extend(delta.tolist())
        checks.append({"sample_id": row.get("sample_id", row.get("prompt_id")), "task_type": row["task_type"],
                       "response_tokens": len(response_ids), "prompt_tokens": len(prompt_ids),
                       "first_response_action_index": start, "token_mask_alignment": True,
                       "vllm_first_raw_logprob": row["behavior_logprobs"][0], "training_first_raw_logprob": values[0]})
        del output, logits
    absolute = torch.tensor(deltas).abs()
    if not absolute.numel(): raise ValueError("No logprob evidence available")
    # Thresholds are explicit engineering gates, not equality claims or estimated KL.
    mean_limit, p99_limit = ((0.10, 0.60) if mode == "bf16_lora" else (0.50, 3.0))
    result = {"status": "passed" if absolute.mean() <= mean_limit and absolute.quantile(0.99) <= p99_limit else "blocked_policy_distribution_mismatch",
              "training_mode": mode, "rollout_mode": "bf16", "scope": "one_actual_rollout_per_task",
              "tokens_compared": len(deltas), "mean_abs_logprob_delta": float(absolute.mean()),
              "p99_abs_logprob_delta": float(absolute.quantile(0.99)), "max_abs_logprob_delta": float(absolute.max()),
              "signed_mean_logprob_delta": float(torch.tensor(deltas).mean()), "mean_limit": mean_limit,
              "p99_limit": p99_limit, "quantization_difference_expected": mode == "nf4_qlora", "checks": checks}
    _write_stage_json(RESULT_DIR / f"alignment_{mode}.json", result)
    return result


### vLLM 0.11 原始概率与采样语义
显式关闭 top-k/top-p 裁剪、惩罚和模型 generation_config 默认值，使用 T=1。V1 raw_logprobs 才与行为采样分布相同；不从 top-1 代替缺失的采样 Token。


In [ ]:
def _generate_rollout_records(prompt_rows: list[dict], model_source: str, policy_version: str, adapter_path: str, n: int) -> list[dict]:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    tokenizer = AutoTokenizer.from_pretrained(model_source, local_files_only=Path(model_source).exists())
    engine = None
    try:
        engine = LLM(model=model_source, tokenizer=model_source, dtype="bfloat16", enable_lora=True,
                     tensor_parallel_size=1, gpu_memory_utilization=float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.80")),
                     max_model_len=MAX_MODEL_LEN, max_num_seqs=8, enforce_eager=True, generation_config="vllm",
                     logprobs_mode="raw_logprobs", seed=SEED)
        sampling = SamplingParams(temperature=1.0, top_p=1.0, top_k=-1, repetition_penalty=1.0,
                                  presence_penalty=0.0, frequency_penalty=0.0, n=n, max_tokens=MAX_NEW_TOKENS,
                                  logprobs=1, seed=SEED, skip_special_tokens=False)
        rendered = [tokenizer.apply_chat_template([{"role":"user","content":r["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=True) for r in prompt_rows]
        if any(len(tokenizer.encode(s, add_special_tokens=False)) + MAX_NEW_TOKENS > MAX_MODEL_LEN for s in rendered):
            raise ValueError(f"Rollout context budget exceeds max_model_len={MAX_MODEL_LEN}")
        outputs = engine.generate(rendered, sampling, lora_request=LoRARequest("sft", 1, adapter_path), use_tqdm=True)
        if len(outputs) != len(prompt_rows): raise ValueError("Rollout request count mismatch")
        rows = []
        for original, request in zip(prompt_rows, outputs):
            if len(request.outputs) != n: raise ValueError("Rollout group size mismatch")
            prompt_ids = list(request.prompt_token_ids or [])
            for index, out in enumerate(request.outputs):
                ids = list(out.token_ids or [])
                rows.append({"sample_id":str(original["id"]), "prompt_id":str(original["id"]),
                    "task_type":original["task_type"], "prompt":original["prompt"], "expected_answer":str(original["answer"]),
                    "completion":out.text, "prompt_token_ids":prompt_ids, "completion_token_ids":ids,
                    "response_mask":[0]*len(prompt_ids)+[1]*len(ids), "behavior_logprobs":_chosen_logprob_values(out),
                    "finish_reason":out.finish_reason, "stop_reason":out.stop_reason, "truncated":out.finish_reason=="length",
                    "policy_version":policy_version, "temperature":1.0, "top_p":1.0, "top_k":-1,
                    "group_index":index, "source":"vllm_rollout", "fixture":False, "trace_verified":False})
        return rows
    finally:
        if engine is not None: del engine
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### Adapter 磁盘重载检查
更新后先保存，再从磁盘加载为独立 PEFT Adapter，比较真实模型 logits；不把目录存在等同于重载成功。


In [ ]:
def actual_adapter_reload(model, adapter_dir: Path, input_ids: list[int]) -> dict:
    """Load saved PEFT weights from disk under a second adapter and compare logits."""
    model.eval()
    original = model.active_adapter
    if isinstance(original, list): original = original[0]
    device = model.get_input_embeddings().weight.device
    probe = torch.tensor([input_ids[:32]], device=device)
    with torch.no_grad():
        before = model(input_ids=probe, attention_mask=torch.ones_like(probe), use_cache=False).logits.float().cpu()
    model.load_adapter(str(adapter_dir), adapter_name="disk_reload_check", is_trainable=False)
    model.set_adapter("disk_reload_check")
    with torch.no_grad():
        after = model(input_ids=probe, attention_mask=torch.ones_like(probe), use_cache=False).logits.float().cpu()
    maximum = float((before - after).abs().max())
    model.set_adapter(original)
    if maximum > 1e-5: raise ValueError(f"Adapter reload logit mismatch: {maximum}")
    return {"status":"passed", "method":"PEFT load_adapter from saved disk weights, compare real-model logits", "max_abs_logit_difference":maximum}


### 来源检查：Train 原题和候选索引
持久化与重新读取时核对原始 Train ID、Prompt、答案和任务；不读取 Holdout。


In [ ]:
def _check_train_origin(records):
    import pandas as pd
    frame = pd.read_csv(TRAIN_PATH,dtype=str,keep_default_na=False)
    if set(frame["split"]) != {"train"}: raise ValueError("Train split metadata mismatch")
    lookup = frame.set_index("id").to_dict("index")
    for row in records:
        original = lookup.get(str(row["sample_id"]))
        if original is None or row["prompt"] != original["prompt"] or str(row["expected_answer"]) != original["answer"] or row["task_type"] != original["task_type"]:
            raise ValueError("Rollout is not an unchanged official Train instance")


## GPU Smoke 独立执行入口：DPO
Rollout Kernel 只设置 `RUN_DPO_ROLLOUT=1`；退出进程确认显存释放后，在训练 Kernel 只设置 `RUN_DPO_TRAINING=1`。两次执行使用相同 `GPU_SMOKE_RUN_ID`、数据目录和 Notebook 03 的 SFT Manifest。


In [ ]:
if RUN_DPO_ROLLOUT and RUN_DPO_TRAINING:
    dpo_gpu_status = {"status":"blocked_kernel_switch_required", "reason":"分别在 vLLM 和训练环境执行；两阶段开关不能同时开启"}
elif RUN_DPO_ROLLOUT:
    dpo_gpu_status = run_dpo_rollout_phase()
elif RUN_DPO_TRAINING:
    dpo_gpu_status = run_dpo_training_phase(os.environ.get("RL_QUANTIZATION_MODE", "nf4_qlora"))
else:
    dpo_gpu_status = {"status":"skipped_by_default"}
print(json.dumps(dpo_gpu_status,ensure_ascii=False,indent=2))
